# Inferential Statistics - Shipping Performance

## Purpose

This notebook tests whether shipping-related patterns observed in the numeric and visualization notebooks
reflect statistically meaningful differences in business outcomes.

Based on descriptive results:
- Shipping cost vs shipping time correlation is weak and slightly negative (~ -0.143), suggesting cost is not simply “longer = more expensive.” 
- Shipping delay vs profit correlation is essentially zero (~ 0.0015), suggesting delay alone is not a strong linear driver of profitability. 
- Profit margins differ across ship modes in the visual analysis. 

Inferential questions:
1) Do profit margins differ across ship modes?
2) Does shipping cost predict profit (and does this vary by ship mode)?
3) Do ship mode / delay / cost explain return probability?

## Contents
1. [Setup and data preparation](#1-setup-and-data-preparation)
2. [Profit margin differences by ship mode](#2-profit-margin-differences-by-ship-mode)
3. [Profit vs shipping cost with ship mode controls](#3-profit-vs-shipping-cost-with-ship-mode-controls)
4. [Return probability using ship mode, delay, and shipping cost](#4-return-probability-using-ship-mode-delay-and-shipping-cost)
5. [Interpretation](#5-interpretation) 

## 1. Setup and Data Preparation

In [1]:
# Load required libraries
library(tidyverse)
library(janitor)
library(dplyr)
library(ggplot2)
library(skimr)
library(purrr)
library(lubridate)

# Source helper scripts
source("../../R/apply_factors.R")
source("../../R/analysis_helpers.R")
source("../../R/temporal_helpers.R")

# Load data
tables <- list(
  Orders  = readr::read_csv("../../data/processed/Orders.csv"),
  Returns = readr::read_csv("../../data/processed/Returns.csv"),
  People  = readr::read_csv("../../data/processed/People.csv")
)

# Apply factor transformations
tables <- apply_factors(tables)

# Extract tables
orders  <- tables$Orders
returns <- tables$Returns
people  <- tables$People

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test


Rows: 51290 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (13): order_id, ship_mode, customer_name, segment, state, country, mark...
dbl   (6): sales, quantity, discount, profit, shipping_cost, year
date  (2): order_date, ship_date

ℹ Use `spec()` to retrieve the full column specification f

In [2]:
orders <- orders |>
    mutate(
        shipping_time_days = as.numeric(ship_date - order_date),
        margin = if_else(sales > 0, profit / sales, NA_real_)
    )

returns_one <- returns |>
    transmute(order_id, returned = as.integer(returned)) |>
    group_by(order_id) |>
    summarise(returned = as.integer(any(returned == TRUE)), .groups = "drop")

orders_inf <- orders |>
    left_join(returns_one, by = "order_id") |>
    mutate(returned = replace_na(returned, 0L))

## 2. Profit margin differences by ship mode

### Model (One-way ANOVA)
We model profit margin as a function of shipping mode:

$$
\text{margin}_i = \mu + \alpha_{\text{ship\_mode}(i)} + \varepsilon_i
$$

Where:
- $\text{margin}_i$ is the profit margin for order $i$
- $\mu$ is the overall mean margin
- $\alpha_{\text{ship\_mode}(i)}$ is the effect of the ship mode used for order $i$
- $\varepsilon_i$ is the error term

### Hypotheses
- **$H_0$:** All ship modes have the same mean margin  

$$
\alpha_1 = \alpha_2 = \cdots = \alpha_K = 0
$$

(equivalently: all ship-mode mean margins are equal)

- **$H_1$:** At least one ship-mode mean differs  

$$
\exists\, j \text{ such that } \alpha_j \neq 0
$$

In [3]:
## margin ~ ship_mode
m1_anova_margin_shipmode <- aov(margin ~ ship_mode, data = orders_inf)
summary(m1_anova_margin_shipmode)

# Post-hoc to identify which modes differ
m1_tukey <- TukeyHSD(m1_anova_margin_shipmode)
m1_tukey

               Df Sum Sq Mean Sq F value Pr(>F)  
ship_mode       3      2  0.5464    2.52 0.0561 .
Residuals   51286  11120  0.2168                 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = margin ~ ship_mode, data = orders_inf)

$ship_mode
                                     diff           lwr        upr     p adj
Standard Class-First Class   0.0136632423 -0.0017373970 0.02906388 0.1028656
Second Class-First Class     0.0183545346  0.0002025435 0.03650653 0.0463148
Same Day-First Class         0.0178383805 -0.0090037699 0.04468053 0.3197795
Second Class-Standard Class  0.0046912923 -0.0089217823 0.01830437 0.8125005
Same Day-Standard Class      0.0041751382 -0.0198315703 0.02818185 0.9702574
Same Day-Second Class       -0.0005161541 -0.0263741730 0.02534186 0.9999516


### Interpretation 

The ANOVA tests whether average profit margins are equal across ship modes.  
- If the p-value is small, we reject the null and conclude that margin differs across at least one ship mode.

Tukey post-hoc results identify which ship-mode pairs differ while controlling for multiple comparisons.

Business meaning:
Ship mode is not purely operational—some fulfillment choices appear structurally more/less “efficient” in margin terms.
This does not prove causality (ship mode is correlated with geography, priority, product mix), but it flags where
shipping strategy may be economically important.

## 3. Profit vs shipping cost with ship mode controls

### Model (Linear regression with interaction)
We model profit using shipping cost and ship mode, allowing the shipping-cost effect to vary by ship mode:

$$
\text{profit}_i =
\beta_0
+ \beta_1 \cdot \text{shipping\_cost}_i
+ \gamma_{\text{ship\_mode}(i)}
+ \delta_{\text{ship\_mode}(i)} \cdot \text{shipping\_cost}_i
+ \varepsilon_i
$$

Where:
- $\text{profit}_i$ is the profit for order $i$
- $\text{shipping\_cost}_i$ is the shipping cost for order $i$
- $\gamma_{\text{ship\_mode}(i)}$ captures ship-mode baseline differences (relative to a reference ship mode)
- $\delta_{\text{ship\_mode}(i)}$ captures how the shipping-cost slope differs by ship mode
- $\varepsilon_i$ is the error term

### Hypotheses

**Effect of shipping cost in the baseline ship mode**

- **$H_0$:** $\beta_1 = 0$ (shipping cost has no association with profit in the reference ship mode)  
- **$H_1$:** $\beta_1 \neq 0$

**Ship-mode differences in baseline profit**

- **$H_0$:** $\gamma_j = 0$ for a given ship mode $j$  
- **$H_1$:** $\gamma_j \neq 0$

**Ship-mode differences in shipping-cost sensitivity**

- **$H_0$:** $\delta_j = 0$ for a given ship mode $j$  
- **$H_1$:** $\delta_j \neq 0$

In [4]:
## profit ~ shipping_cost * ship_mode
m2_profit_cost_shipmode <- lm(profit ~ shipping_cost * ship_mode, data = orders_inf)
summary(m2_profit_cost_shipmode)


Call:
lm(formula = profit ~ shipping_cost * ship_mode, data = orders_inf)

Residuals:
    Min      1Q  Median      3Q     Max 
-7399.2    -9.5     6.1    19.7  7783.8 

Coefficients:
                                      Estimate Std. Error t value Pr(>|t|)    
(Intercept)                           -2.20984    2.07713  -1.064   0.2874    
shipping_cost                          0.72927    0.02297  31.750  < 2e-16 ***
ship_modeStandard Class               -4.48928    2.31208  -1.942   0.0522 .  
ship_modeSecond Class                  0.11622    2.71223   0.043   0.9658    
ship_modeSame Day                      7.87745    4.02561   1.957   0.0504 .  
shipping_cost:ship_modeStandard Class  1.05516    0.03192  33.059  < 2e-16 ***
shipping_cost:ship_modeSecond Class    0.27593    0.03344   8.253  < 2e-16 ***
shipping_cost:ship_modeSame Day       -0.20445    0.04260  -4.799  1.6e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 160.3 on 512

### Interpretation 

The main shipping_cost coefficient describes the profit change per unit of shipping cost in the reference ship mode.
Interaction terms show whether the cost→profit relationship differs for other ship modes.

How to read:
- A negative shipping_cost coefficient suggests that, within a ship mode, more expensive shipments are associated
  with lower profit (cost pressure not fully offset by higher sales/profit).
- Significant interactions imply that this cost pressure is stronger/weaker depending on the ship mode.

Business meaning:
This model turns the descriptive “shipping cost structure matters” insight into a testable claim:
shipping cost can be economically harmful in some fulfillment paths, even if delay itself is not. 

## 4. Return probability using ship mode, delay, and shipping cost

### Model (Logistic regression)
We model return probability using a logistic regression with ship mode and continuous shipping features:

$$
\Pr(\text{returned}_i = 1)
=
\text{logit}^{-1}
\left(
\beta_0
+ \beta_1 \cdot \text{shipping\_time}_i
+ \beta_2 \cdot \text{shipping\_cost}_i
+ \gamma_{\text{ship\_mode}(i)}
\right)
$$

Equivalently, in log-odds form:

$$
\log
\left(
\frac{\Pr(\text{returned}_i = 1)}
{1 - \Pr(\text{returned}_i = 1)}
\right)
=
\beta_0
+ \beta_1 \cdot \text{shipping\_time}_i
+ \beta_2 \cdot \text{shipping\_cost}_i
+ \gamma_{\text{ship\_mode}(i)}
$$

Where:
- $\text{returned}_i \in \{0,1\}$ indicates whether order $i$ was returned
- $\text{shipping\_time}_i$ is the shipping time for order $i$
- $\text{shipping\_cost}_i$ is the shipping cost for order $i$
- $\gamma_{\text{ship\_mode}(i)}$ captures ship-mode differences relative to a reference ship mode
- $\beta_0$ is the baseline log-odds

### Hypotheses

**Effect of shipping time**

- **$H_0$:** $\beta_1 = 0$ (shipping time does not affect return odds)  
- **$H_1$:** $\beta_1 \neq 0$

**Effect of shipping cost**

- **$H_0$:** $\beta_2 = 0$ (shipping cost does not affect return odds)  
- **$H_1$:** $\beta_2 \neq 0$

**Ship-mode effects (per coefficient)**

- **$H_0$ (per ship mode):**

$$
\gamma_j = 0
\quad \Longleftrightarrow \quad
\exp(\gamma_j) = 1
$$

- **$H_1$:**

$$
\exists\, j \text{ such that } \gamma_j \neq 0
\quad \Longleftrightarrow \quad
\exp(\gamma_j) \neq 1
$$

In [5]:
## returned ~ ship_mode + shipping_time_days + shipping_cost
m3_return_logit <- glm(
  returned ~ ship_mode + shipping_time_days + shipping_cost,
  data = orders_inf,
  family = binomial(link = "logit")
)
summary(m3_return_logit)

# Optional: odds ratios + CI for interpretation
library(broom)
tidy(m3_return_logit, exponentiate = TRUE, conf.int = TRUE) |>
  arrange(desc(estimate))


Call:
glm(formula = returned ~ ship_mode + shipping_time_days + shipping_cost, 
    family = binomial(link = "logit"), data = orders_inf)

Coefficients:
                          Estimate Std. Error z value Pr(>|z|)    
(Intercept)             -2.7359242  0.0643371 -42.525  < 2e-16 ***
ship_modeStandard Class -0.1984123  0.0754225  -2.631  0.00852 ** 
ship_modeSecond Class   -0.0839759  0.0653514  -1.285  0.19880    
ship_modeSame Day        0.0217443  0.1001432   0.217  0.82811    
shipping_time_days       0.0223998  0.0190231   1.178  0.23899    
shipping_cost            0.0006938  0.0002939   2.361  0.01822 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 23131  on 51289  degrees of freedom
Residual deviance: 23115  on 51284  degrees of freedom
AIC: 23127

Number of Fisher Scoring iterations: 5


term,estimate,std.error,statistic,p.value,conf.low,conf.high
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
shipping_time_days,1.02265259,0.0190231458,1.1775038,0.238994493,0.98513534,1.06141190
ship_modeSame Day,1.02198246,0.1001432482,0.2171323,0.828105242,0.83822371,1.24143935
shipping_cost,1.00069404,0.0002938553,2.3610372,0.018223903,1.00010116,1.00125480
ship_modeSecond Class,0.91945345,0.0653514255,-1.2849889,0.198796157,0.80905690,1.04534264
ship_modeStandard Class,0.82003167,0.0754225483,-2.6306764,0.008521514,0.70762225,0.95107725
(Intercept),0.06483406,0.0643370588,-42.5248573,0.000000000,0.05711078,0.07349519


### Interpretation 

This model estimates how return likelihood changes with ship mode, shipping time, and shipping cost.
Exponentiated coefficients are odds ratios (OR):
- OR > 1 means higher return odds
- OR < 1 means lower return odds

Expected outcome given the numeric notebook:
Because delay–profit correlation is ~0 and delay buckets show only small return-rate differences, we may find that
shipping_time_days is weak or not significant once ship mode and cost are included. :contentReference[oaicite:9]{index=9}

Business meaning:
If ship mode is significant but delay is not, return risk is more about the fulfillment *path* (how we ship)
than the raw transit duration. If shipping_cost is significant, it may proxy for difficult shipments that are also
more return-prone (fragility, distance, complexity).

## 5. Interpretation

- Model 1 tests whether ship modes differ in margin efficiency (ANOVA + post-hoc).
- Model 2 tests whether shipping cost predicts profit and whether that relationship varies by ship mode.
- Model 3 tests whether ship mode, delay, and cost explain return probability.

Overall framing consistent with descriptive findings: Delay alone does not appear to be a dominant driver of economic outcomes, while shipping cost structure and ship-mode
choice are more likely to matter operationally and financially. 